# 01 -- Data Acquisition

Pull 2025 MLB Statcast pitch-level data from Baseball Savant via `pybaseball`.

**What we're getting:** Every pitch thrown in the 2025 regular season (~700K+ pitches), with full Statcast tracking data: velocity, spin, movement, plate location, game state, and outcome (delta run expectancy).

**Why parquet:** Fast columnar reads, smaller on disk than CSV, preserves dtypes. Same reason your HOOD project stores intermediate data in structured formats.

In [1]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")

import pandas as pd
import numpy as np

from src.data import pull_statcast_season, clean_statcast, save_parquet, get_data

## Pull the data

This takes 5-15 minutes on a first run (API rate limits). Subsequent runs load from the parquet cache in seconds.

> If the API is slow or flaky, `pybaseball` will retry automatically. The chunked pull in `data.py` keeps each request under the 40-day limit.

In [2]:
# First run: pulls from API and caches as parquet
# Subsequent runs: loads from cache instantly
df = get_data(year=2025, force_refresh=False)
print(f"Loaded {len(df):,} pitches")
print(f"Date range: {df['game_date'].min().date()} to {df['game_date'].max().date()}")
print(f"Unique pitchers: {df['pitcher'].nunique():,}")
print(f"Unique batters: {df['batter'].nunique():,}")

2026-03-03 12:47:32,766 | No cache found, pulling from Statcast API...
2026-03-03 12:47:32,767 | Pulling Statcast data: 2025-03-20 to 2025-10-01
2026-03-03 12:47:32,768 |   Chunk: 2025-03-20 to 2025-04-02


This is a large query, it may take a moment to complete


100%|██████████| 14/14 [00:05<00:00,  2.51it/s]
2026-03-03 12:47:38,654 |   Chunk: 2025-04-03 to 2025-04-16


This is a large query, it may take a moment to complete


100%|██████████| 14/14 [00:01<00:00,  7.02it/s]
2026-03-03 12:47:40,688 |   Chunk: 2025-04-17 to 2025-04-30


...

This is a large query, it may take a moment to complete


100%|██████████| 14/14 [00:02<00:00,  6.15it/s]
2026-03-03 12:48:06,926 |   Chunk: 2025-09-18 to 2025-10-01


This is a large query, it may take a moment to complete


100%|██████████| 14/14 [00:01<00:00,  7.06it/s]
2026-03-03 12:48:09,235 | Raw pull: 737,186 pitches
2026-03-03 12:48:09,470 | After regular season filter: 711,897 pitches
2026-03-03 12:48:09,542 | Dropped 2,753 rows with missing critical fields
2026-03-03 12:48:09,583 | Clean data: 709,144 pitches (kept 96.2%)
2026-03-03 12:48:09,852 | Saved to /Users/angelrios/PycharmProjects/PythonProject/statcast-bayesian-pitch-model/data/statcast_2025.parquet (24.4 MB)


Loaded 709,144 pitches
Date range: 2025-03-27 to 2025-09-28
Unique pitchers: 873
Unique batters: 673


## Data quality check

Before we touch features or modeling, verify the data is clean. This is the Costco supervisor in you: inspect what you expect.

In [3]:
# Missing values in key columns
key_cols = [
    "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
    "plate_x", "plate_z", "delta_run_exp", "pitch_type",
    "balls", "strikes", "stand", "p_throws",
]
missing = df[key_cols].isnull().sum()
print("Missing values:")
print(missing[missing > 0] if missing.any() else "None -- clean.")

Missing values:
release_spin_rate    1760
dtype: int64


In [4]:
# Distribution of pitch types
print("\nPitch type counts:")
print(df["pitch_type"].value_counts().to_string())


Pitch type counts:
pitch_type
FF    225957
SI    110195
SL    104905
CH     72987
FC     54138
ST     50079
CU     48730
FS     23554
KC     12084
SV      3525
EP       991
FA       903
FO       540
CS       355
KN       136
PO        54
SC         7
UN         4


In [5]:
# Quick sanity check on delta_run_exp
# Should be centered near zero with tails in both directions
print("\ndelta_run_exp summary:")
print(df["delta_run_exp"].describe().round(4))


delta_run_exp summary:
count    709144.0
mean       0.0001
std        0.2252
min        -0.577
25%        -0.062
50%        -0.038
75%         0.039
max         2.696
Name: delta_run_exp, dtype: Float64


In [6]:
# Pitches per pitcher distribution
pitcher_counts = df.groupby("pitcher").size()
print(f"\nPitches per pitcher:")
print(f"  Median: {pitcher_counts.median():.0f}")
print(f"  Mean:   {pitcher_counts.mean():.0f}")
print(f"  Min:    {pitcher_counts.min()}")
print(f"  Max:    {pitcher_counts.max():,}")
print(f"  <100 pitches: {(pitcher_counts < 100).sum()} pitchers")
print(f"  >2000 pitches: {(pitcher_counts > 2000).sum()} pitchers")
print()
print("This is why partial pooling matters: pitchers with <100 pitches")
print("will get shrunk toward the league average. The model borrows strength.")


Pitches per pitcher:
  Median: 599
  Mean:   812
  Min:    2
  Max:    3,282
  <100 pitches: 147 pitchers
  >2000 pitches: 103 pitchers

This is why partial pooling matters: pitchers with <100 pitches
will get shrunk toward the league average. The model borrows strength.


In [7]:
# Physical sanity checks
print("Velocity range:", df["release_speed"].min(), "-", df["release_speed"].max(), "mph")
print("Spin rate range:", df["release_spin_rate"].min(), "-", df["release_spin_rate"].max(), "rpm")
print("Plate X range:", df["plate_x"].min().round(2), "-", df["plate_x"].max().round(2), "ft")
print("Plate Z range:", df["plate_z"].min().round(2), "-", df["plate_z"].max().round(2), "ft")

Velocity range: 31.1 - 104.2 mph
Spin rate range: 42 - 3547 rpm
Plate X range: -7.14 - 5.7 ft
Plate Z range: -18.48 - 10.42 ft


## Summary

Data is pulled, cleaned, and cached. Key takeaways:
- ~700K pitches from the 2025 regular season
- Nulls dropped in critical columns (pitch physics + outcome)
- Base runners encoded as binary occupied flags
- Wide range of pitcher sample sizes (partial pooling will handle this)

Next: `02_eda_and_feature_engineering.ipynb` for visualization and feature creation.